In [1]:
with open("mini_gpt/data/databook.txt", "r", encoding="utf-8") as file:
    text = file.read()

print(f"Loaded {len(text):,} characters")
print(text[:500])

Loaded 3,227,613 characters
The Project Gutenberg eBook of War and Peace
    
This eBook is for the use of anyone anywhere in the United States and
most other parts of the world at no cost and with almost no restrictions
whatsoever. You may copy it, give it away or re-use it under the terms
of the Project Gutenberg License included with this eBook or online
at www.gutenberg.org. If you are not located in the United States,
you will have to check the laws of the country where you are located
before using this eBook.

Title:


In [2]:
import re

lines = text.splitlines()
chapter_titles = re.findall(r"(?m)^CHAPTER [IVXLCDM]+$", text)

print(f"Characters: {len(text):,}")
print(f"Lines: {len(lines):,}")
print(f"Words: {len(text.split()):,}")
print(f"Chapter headings found: {len(chapter_titles):,}")
print("First chapter headings:", chapter_titles[:5])
print("\n--- Beginning ---")
print(text[:1_000])
print("\n--- End ---")
print(text[-500:])

Characters: 3,227,613
Lines: 66,040
Words: 566,333
Chapter headings found: 365
First chapter headings: ['CHAPTER I', 'CHAPTER II', 'CHAPTER III', 'CHAPTER IV', 'CHAPTER V']

--- Beginning ---
The Project Gutenberg eBook of War and Peace
    
This eBook is for the use of anyone anywhere in the United States and
most other parts of the world at no cost and with almost no restrictions
whatsoever. You may copy it, give it away or re-use it under the terms
of the Project Gutenberg License included with this eBook or online
at www.gutenberg.org. If you are not located in the United States,
you will have to check the laws of the country where you are located
before using this eBook.

Title: War and Peace

Author: graf Leo Tolstoy

Translator: Aylmer Maude
        Louise Maude


        
Release date: April 1, 2001 [eBook #2600]
                Most recently updated: June 14, 2022

Language: English

Other information and formats: www.gutenberg.org/ebooks/2600

Credits: An Anonymous Volunteer 

In [3]:
chars = sorted(set(text))
vocab_size = len(chars)

print(f"Unique characters ({vocab_size}):")
print(" ".join(repr(char) for char in chars))

Unique characters (112):
'\n' ' ' '!' '#' '$' '%' '(' ')' '*' '+' ',' '-' '.' '/' '0' '1' '2' '3' '4' '5' '6' '7' '8' '9' ':' ';' '=' '?' 'A' 'B' 'C' 'D' 'E' 'F' 'G' 'H' 'I' 'J' 'K' 'L' 'M' 'N' 'O' 'P' 'Q' 'R' 'S' 'T' 'U' 'V' 'W' 'X' 'Y' 'Z' '[' ']' 'a' 'b' 'c' 'd' 'e' 'f' 'g' 'h' 'i' 'j' 'k' 'l' 'm' 'n' 'o' 'p' 'q' 'r' 's' 't' 'u' 'v' 'w' 'x' 'y' 'z' 'À' 'Á' 'É' 'à' 'á' 'â' 'ä' 'æ' 'ç' 'è' 'é' 'ê' 'ë' 'í' 'î' 'ï' 'ó' 'ô' 'ö' 'ú' 'ü' 'ý' 'œ' '—' '‘' '’' '“' '”' '•' '™'


In [4]:
stoi = {char: index for index, char in enumerate(chars)}
itos = {index: char for char, index in stoi.items()}

print("Character to integer:", dict(list(stoi.items())[:20]))
print("Integer to character:", dict(list(itos.items())[:20]))

Character to integer: {'\n': 0, ' ': 1, '!': 2, '#': 3, '$': 4, '%': 5, '(': 6, ')': 7, '*': 8, '+': 9, ',': 10, '-': 11, '.': 12, '/': 13, '0': 14, '1': 15, '2': 16, '3': 17, '4': 18, '5': 19}
Integer to character: {0: '\n', 1: ' ', 2: '!', 3: '#', 4: '$', 5: '%', 6: '(', 7: ')', 8: '*', 9: '+', 10: ',', 11: '-', 12: '.', 13: '/', 14: '0', 15: '1', 16: '2', 17: '3', 18: '4', 19: '5'}


In [9]:
import sys
from pathlib import Path

venv_site_packages = Path(".venv/Lib/site-packages").resolve()
if venv_site_packages.is_dir() and str(venv_site_packages) not in sys.path:
    sys.path.insert(0, str(venv_site_packages))

import torch

data = torch.tensor([stoi[character] for character in text], dtype=torch.long)

print(f"Tensor shape: {data.shape}")
print(f"Tensor dtype: {data.dtype}")
print("First 20 token IDs:", data[:20])

Tensor shape: torch.Size([3227613])
Tensor dtype: torch.int64
First 20 token IDs: tensor([47, 63, 60,  1, 43, 73, 70, 65, 60, 58, 75,  1, 34, 76, 75, 60, 69, 57,
        60, 73])


In [10]:
split_idx = int(0.9 * len(data))
train_data = data[:split_idx]
val_data = data[split_idx:]

print(f"Training tokens: {len(train_data):,}")
print(f"Validation tokens: {len(val_data):,}")

Training tokens: 2,904,851
Validation tokens: 322,762


In [11]:
batch_size = 4
block_size = 8
device = "cuda" if torch.cuda.is_available() else "cpu"


def get_batch(split):
    source = {"train": train_data, "val": val_data}[split]
    starts = torch.randint(len(source) - block_size, (batch_size,))
    x = torch.stack([source[start:start + block_size] for start in starts])
    y = torch.stack([source[start + 1:start + block_size + 1] for start in starts])
    return x.to(device), y.to(device)


xb, yb = get_batch("train")
xv, yv = get_batch("val")

print(f"Device: {device}")
print(f"Train batch shapes: inputs={xb.shape}, targets={yb.shape}")
print(f"Validation batch shapes: inputs={xv.shape}, targets={yv.shape}")
print("Example input IDs:  ", xb[0])
print("Example target IDs: ", yb[0])

Device: cpu
Train batch shapes: inputs=torch.Size([4, 8]), targets=torch.Size([4, 8])
Validation batch shapes: inputs=torch.Size([4, 8]), targets=torch.Size([4, 8])
Example input IDs:   tensor([ 73,  12,   1, 108,  36,   1,  66,  69])
Example target IDs:  tensor([ 12,   1, 108,  36,   1,  66,  69,  70])


In [12]:
from mini_gpt.model import BigramLanguageModel

model = BigramLanguageModel(vocab_size).to(device)
logits, loss = model(xb, yb)

print(f"Logits shape: {logits.shape}")
print(f"Initial next-character loss: {loss.item():.4f}")

Logits shape: torch.Size([4, 8, 112])
Initial next-character loss: 5.4482


In [13]:
batch_size = 64
block_size = 64
learning_rate = 1e-2
max_iters = 1_000
eval_iters = 20


@torch.no_grad()
def estimate_loss():
    model.eval()
    losses = {}
    for split in ("train", "val"):
        split_losses = torch.zeros(eval_iters)
        for step in range(eval_iters):
            inputs, targets = get_batch(split)
            _, batch_loss = model(inputs, targets)
            split_losses[step] = batch_loss.item()
        losses[split] = split_losses.mean().item()
    model.train()
    return losses


model = BigramLanguageModel(vocab_size).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
initial_losses = estimate_loss()

for step in range(max_iters):
    inputs, targets = get_batch("train")
    _, batch_loss = model(inputs, targets)
    optimizer.zero_grad(set_to_none=True)
    batch_loss.backward()
    optimizer.step()

    if (step + 1) % 200 == 0:
        print(f"Step {step + 1}: train loss {batch_loss.item():.4f}")

final_losses = estimate_loss()
print(f"Initial train/validation loss: {initial_losses['train']:.4f} / {initial_losses['val']:.4f}")
print(f"Final train/validation loss:   {final_losses['train']:.4f} / {final_losses['val']:.4f}")

Step 200: train loss 3.1467
Step 400: train loss 2.6337
Step 600: train loss 2.5100
Step 800: train loss 2.4943
Step 1000: train loss 2.4552
Initial train/validation loss: 5.2858 / 5.2915
Final train/validation loss:   2.4850 / 2.5133


In [14]:
start = torch.tensor([[stoi["\n"]]], dtype=torch.long, device=device)
generated_ids = model.generate(start, max_new_tokens=500)[0].tolist()
generated_text = "".join(itos[token_id] for token_id in generated_ids)

print(generated_text)


 ther be wenthe
Kuenghino18hiCally m whe

avan s St g aze ispe Buf a bathide tein. he wat toweng t tîBacere s’sas, aces l blk sk Moberele hertindo coteashbe te r wry]e ad thory farorere’se grdaithache 7 an ferofintin olled le  Mit wissecaslosoos Ve.
lier Fod iep toke igh w, Bevin diotlidlato oildlabuld?” thowióveshen’towad


Bris ly hoftheysteý—Nate t wht No o Of ofofavispe gng m ore clemer oídrita llèriroumed œ$Anth w f pusus tl hellentofisenghesaseret heöä’tuaichis, thethateet VI tr the ngrtht
